In [1]:
import numpy as np
import pandas as pd
SEED = 7
WINDOW = 4
fs = 8
print(WINDOW / fs) # seconds per window
print(type(fs), type(WINDOW / fs))

0.5
<class 'int'> <class 'float'>


In [2]:
data = pd.DataFrame({
'record_id': ['R1', 'R2', 'R3'],
'machine_id': ['M1', 'M1', 'M2'],
'rul_hours': [15., 10., 5.],
'sample_rate_hz': [8, 8, 8],
'sample_0002': [3., 2., 0.],
'sample_0000': [1., 2., 4.],
'sample_0003': [4., 1., 2.],
'sample_0001': [2., 1., 3.],
'sample_0004': [5., 3., 5.],
'sample_0005': [6., 2., 1.],
'sample_0006': [7., 4., 0.],
'sample_0007': [8., 3., 2.],
})
print(data.shape)
print(data.columns.tolist())
print(data.head(2))

(3, 12)
['record_id', 'machine_id', 'rul_hours', 'sample_rate_hz', 'sample_0002', 'sample_0000', 'sample_0003', 'sample_0001', 'sample_0004', 'sample_0005', 'sample_0006', 'sample_0007']
  record_id machine_id  rul_hours  sample_rate_hz  sample_0002  sample_0000  \
0        R1         M1       15.0               8          3.0          1.0   
1        R2         M1       10.0               8          2.0          2.0   

   sample_0003  sample_0001  sample_0004  sample_0005  sample_0006  \
0          4.0          2.0          5.0          6.0          7.0   
1          1.0          1.0          3.0          2.0          4.0   

   sample_0007  
0          8.0  
1          3.0  


In [3]:
print(data['rul_hours'])
print(data[['record_id', 'machine_id']])
print(data['sample_rate_hz'].iloc[0])
sample_cols = sorted(
[c for c in data.columns
if c.startswith('sample_') and c.split('_')[1].isdigit()],
key=lambda c: int(c.split('_')[1])
)
print(sample_cols)

0    15.0
1    10.0
2     5.0
Name: rul_hours, dtype: float64
  record_id machine_id
0        R1         M1
1        R2         M1
2        R3         M2
8
['sample_0000', 'sample_0001', 'sample_0002', 'sample_0003', 'sample_0004', 'sample_0005', 'sample_0006', 'sample_0007']


In [4]:
signals = data[sample_cols].to_numpy(dtype='float32')
meta = data[['record_id','machine_id','rul_hours']].copy()
fs = int(data['sample_rate_hz'].iloc[0])
print(signals.shape, signals.dtype)
print(signals[0])
print(meta.head())
assert len(sample_cols) == 8, 'Expected eight sample columns'
assert data['sample_rate_hz'].nunique() == 1
assert meta.record_id.is_unique
assert np.isfinite(signals).all()
assert meta.rul_hours.ge(0).all()

(3, 8) float32
[1. 2. 3. 4. 5. 6. 7. 8.]
  record_id machine_id  rul_hours
0        R1         M1       15.0
1        R2         M1       10.0
2        R3         M2        5.0


In [5]:
print(signals[0]) # first recording, all samples
print(signals[0, 2]) # third sample in first recording
print(signals[:2, :4]) # first 2 recordings, first 4 samples
print(signals.shape[1]) # samples per recording
print(np.arange(signals.shape[1]) / fs) # time axis

[1. 2. 3. 4. 5. 6. 7. 8.]
3.0
[[1. 2. 3. 4.]
 [2. 1. 2. 1.]]
8
[0.    0.125 0.25  0.375 0.5   0.625 0.75  0.875]


In [7]:
WINDOW = 4
STRIDE = 4
starts = range(0, signals.shape[1] - WINDOW + 1, STRIDE)
print(list(starts))
for signal in signals:
    for start in starts:
        print(signal[start:start + WINDOW])

[0, 4]
[1. 2. 3. 4.]
[5. 6. 7. 8.]
[2. 1. 2. 1.]
[3. 2. 4. 3.]
[4. 3. 0. 2.]
[5. 1. 0. 2.]


In [8]:
X = np.stack([
signal[s:s + WINDOW]
for signal in signals
for s in starts
])
print('X shape:', X.shape)
print('Window 0:', X[0])
print('Window 1:', X[1])
print('First two windows:', X[:2])

X shape: (6, 4)
Window 0: [1. 2. 3. 4.]
Window 1: [5. 6. 7. 8.]
First two windows: [[1. 2. 3. 4.]
 [5. 6. 7. 8.]]


In [9]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(np.stack([a, b]).shape) # (2, 3)
print(np.stack([a, b], axis=1).shape) # (3, 2)
print(np.concatenate([a, b]).shape) # (6,)

(2, 3)
(3, 2)
(6,)


In [10]:
record_rows = np.repeat(np.arange(len(meta)), len(starts))
record_ids = meta.record_id.to_numpy()[record_rows]
y = meta.rul_hours.to_numpy(dtype='float32')[record_rows]
groups = meta.machine_id.to_numpy()[record_rows]
print('rows:', record_rows)
print('IDs:', record_ids)
print('labels:', y)
print('groups:', groups)
assert len(X) == len(y) == len(groups) == len(record_ids)

rows: [0 0 1 1 2 2]
IDs: ['R1' 'R1' 'R2' 'R2' 'R3' 'R3']
labels: [15. 15. 10. 10.  5.  5.]
groups: ['M1' 'M1' 'M1' 'M1' 'M2' 'M2']


In [11]:
from sklearn.model_selection import train_test_split
ids = np.arange(len(X))
train_idx, rest = train_test_split(
ids, test_size=1/3, random_state=7
)
val_idx, test_idx = train_test_split(
rest, test_size=0.5, random_state=7
)
print(train_idx, val_idx, test_idx)
print('train machines:', np.unique(groups[train_idx]))
print('val machines:', np.unique(groups[val_idx]))

[0 2 1 4] [5] [3]
train machines: ['M1' 'M2']
val machines: ['M2']


In [16]:
print(groups)
print(np.unique(groups))
selected_machines = np.unique(groups)[:1]
print(selected_machines)
mask = np.isin(groups, selected_machines)
print(mask)
print(X)
print(X[mask].shape)
print(np.unique(groups[mask]))
# machine_order = np.random.default_rng(SEED).permutation(
# np.unique(groups))
# train_idx = ids[np.isin(groups, machine_order[:12])]

['M1' 'M1' 'M1' 'M1' 'M2' 'M2']
['M1' 'M2']
['M1']
[ True  True  True  True False False]
[[1. 2. 3. 4.]
 [5. 6. 7. 8.]
 [2. 1. 2. 1.]
 [3. 2. 4. 3.]
 [4. 3. 0. 2.]
 [5. 1. 0. 2.]]
(4, 4)
['M1']


In [17]:
print(X.mean()) # one global mean
print(X.mean(axis=1)) # one mean per window
print(X.mean(axis=1, keepdims=True).shape)
X_work = X.copy()
standardised = (X_work - X_work.mean(axis=1, keepdims=True)) / (
        X_work.std(axis=1, keepdims=True) + 1e-8
)
print(standardised.mean(axis=1))

2.9583333
[2.5  6.5  1.5  3.   2.25 2.  ]
(6, 1)
[ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
 -1.4901161e-08 -2.9802322e-08]


In [18]:
X_train = X[train_idx]
X_val = X[val_idx]; X_test = X[test_idx]
train_mean = X_train.mean()
train_std = X_train.std()
# Apply exactly the same reference to all three datasets.
X_train_scaled = (X_train - train_mean) / (train_std + 1e-8)
X_val_scaled = (X_val - train_mean) / (train_std + 1e-8)
X_test_scaled = (X_test - train_mean) / (train_std + 1e-8)
# Compare the per-window spreads before and after scaling.
print(X_train.std(axis=1)[:5])
print(X_train_scaled.std(axis=1)[:5])

[1.118034 0.5      1.118034 1.47902 ]
[0.4986384  0.22299787 0.4986384  0.6596366 ]


In [47]:
X_train = X[train_idx]
X_val = X[val_idx]; X_test = X[test_idx]
train_mean = X_train.mean()
train_std = X_train.std()
# Use the same training-set reference for every split.
X_train_scaled = (X_train - train_mean) / (train_std + 1e-8)
X_val_scaled = (X_val - train_mean) / (train_std + 1e-8)
X_test_scaled = (X_test - train_mean) / (train_std + 1e-8)
# Compare each window's spread before and after.
print(X_train.std(axis=1)[:5])
print(X_train_scaled.std(axis=1)[:5])

[1.118034 0.5      1.118034 1.47902 ]
[0.4986384  0.22299787 0.4986384  0.6596366 ]


In [48]:
def features(windows):
    rms = np.sqrt(np.mean(windows**2, axis=1))
    std = windows.std(axis=1)
    peak_to_peak = np.ptp(windows, axis=1)
    crest = np.max(np.abs(windows), axis=1) / (rms + 1e-8)
    centred = windows - windows.mean(axis=1, keepdims=True)
    kurtosis = np.mean(centred**4, axis=1) / (std**4 + 1e-8)
    
    return np.column_stack([rms, std, peak_to_peak, crest, kurtosis])
F= features(X)
print(F)

[[2.738613   1.118034   3.         1.4605935  1.6399999 ]
 [6.595453   1.118034   3.         1.2129569  1.6399999 ]
 [1.5811388  0.5        1.         1.264911   0.9999999 ]
 [3.082207   0.70710677 2.         1.2977713  2.        ]
 [2.6925824  1.47902    4.         1.4855627  1.8457141 ]
 [2.738613   1.8708287  5.         1.8257418  1.9999999 ]]


In [22]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(F[train_idx])
F_scaled = scaler.transform(F).astype('float32')
print(F_scaled.shape)
print(F_scaled)
print(scaler.mean_)

(6, 5)
[[-0.3489174   0.18253635  0.22941573  0.8801236   0.3413051 ]
 [ 1.6798023   0.18253635  0.22941573 -1.2037863   0.3413051 ]
 [-0.9577553  -1.5729897  -1.6059101  -0.7665819  -1.6705989 ]
 [-0.16818495 -0.98470277 -0.6882472  -0.49005628  1.4730016 ]
 [-0.3731297   1.2079173   1.1470786   1.0902445   0.98798835]
 [-0.3489174   2.3208504   2.0647416   3.9529178   1.4730011 ]]
[3.40194672 1.053772   2.75       1.35600603 1.53142843]


In [23]:
y_scale = float(y[train_idx].max())
y_scaled = y / y_scale
print(y_scaled.shape)
print(y_scaled[:, None].shape)
# CNN: use channels-last for Keras Conv1D
raw_mean = X_work[train_idx].mean()
raw_std = X_work[train_idx].std()
X_cnn = ((X_work - raw_mean) / (raw_std + 1e-8))[:, :, None]
print("CNN shape:", X_cnn.shape)

(6,)
(6, 1)
CNN shape: (6, 4, 1)


In [24]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(F[train_idx])
F_scaled = scaler.transform(F).astype('float32')
print(F_scaled.shape)
print(scaler.mean_)

(6, 5)
[3.40194672 1.053772   2.75       1.35600603 1.53142843]


In [25]:
y_scale = float(y[train_idx].max())
y_scaled = y / y_scale
print(y_scaled.shape)
print(y_scaled[:, None].shape)
# CNN: use channels-last for Keras Conv1D
raw_mean = X_work[train_idx].mean()
raw_std = X_work[train_idx].std()
X_cnn = ((X_work - raw_mean) / (raw_std + 1e-8))[:, :, None]
print("CNN shape:", X_cnn.shape)

(6,)
(6, 1)
CNN shape: (6, 4, 1)


In [28]:
# Run after Sections 1-10; requires tensorflow and scikit-learn.
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error
WIDTH = 16
LR = 0.001
EPOCHS = 30
BATCH_SIZE = 8
keras.utils.set_random_seed(SEED)
def fit_model(model, inputs):
    inputs = np.asarray(inputs, dtype='float32')
    targets = y_scaled[:, None].astype('float32')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='mse'
    )
    stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True)

    history_obj = model.fit(inputs[train_idx], targets[train_idx],validation_data=(inputs[val_idx], targets[val_idx]),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[stopping], verbose=0
    )
    pred_val = model.predict(inputs[val_idx], verbose=0).ravel() * y_scale
    pred_test = model.predict(inputs[test_idx], verbose=0).ravel() * y_scale
    history = np.column_stack([history_obj.history['loss'],history_obj.history['val_loss']])
    return pred_val, pred_test, history

In [29]:
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(SEED)
ann = keras.Sequential([
    keras.Input(shape=(F_scaled.shape[1],)),
    layers.Dense(WIDTH, activation='relu'),
    layers.Dense(1)
])
ann.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 113 (452.00 B)

 Trainable params: 113 (452.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
ann.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='mse'
)
stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
history = ann.fit(
    F_scaled[train_idx], y_scaled[train_idx, None],
    validation_data=(F_scaled[val_idx], y_scaled[val_idx, None]),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[stopping], verbose=0
)

In [31]:
pred_mlp = (ann.predict(F_scaled[test_idx], verbose=0).ravel() * y_scale)

In [42]:
print(pred_mlp)
print(F_scaled)
print(test_idx)
print(train_idx)
print(val_idx)
print(y_scaled*y_scale)

[8.226564]
[[-0.3489174   0.18253635  0.22941573  0.8801236   0.3413051 ]
 [ 1.6798023   0.18253635  0.22941573 -1.2037863   0.3413051 ]
 [-0.9577553  -1.5729897  -1.6059101  -0.7665819  -1.6705989 ]
 [-0.16818495 -0.98470277 -0.6882472  -0.49005628  1.4730016 ]
 [-0.3731297   1.2079173   1.1470786   1.0902445   0.98798835]
 [-0.3489174   2.3208504   2.0647416   3.9529178   1.4730011 ]]
[3]
[0 2 1 4]
[5]
[15. 15. 10. 10.  5.  5.]


In [43]:
raw_mean = X_work[train_idx].mean()
raw_std = X_work[train_idx].std()
X_cnn = ((X_work - raw_mean) / (raw_std + 1e-8))[:, :, None]
X_cnn = X_cnn.astype('float32')
print(X_cnn.shape) # (windows, samples, channels)

(6, 4, 1)


In [44]:
cnn = keras.Sequential([
keras.Input(shape=(WINDOW, 1)),
layers.Conv1D(WIDTH, 9, padding='same', activation='relu'),
layers.MaxPooling1D(pool_size=2),
layers.Conv1D(WIDTH, 9, padding='same', activation='relu'),
layers.GlobalAveragePooling1D(),
layers.Dense(1)
])
cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 4, 16)          │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 2, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 2, 16)          │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,497 (9.75 KB)

 Trainable params: 2,497 (9.75 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
# The notebook reuses fit_model(model, inputs) for both architectures.
val_cnn, pred_cnn, history_cnn = fit_model(cnn, X_cnn)
print('CNN validation MAE (h):',
round(mean_absolute_error(y[val_idx], val_cnn), 2))

CNN validation MAE (h): 0.56


In [46]:
baseline = np.full(len(test_idx), y[train_idx].mean())
for name, pred in [('Mean baseline', baseline),
('ANN', pred_mlp), ('CNN', pred_cnn)]:
    mae = mean_absolute_error(y[test_idx], pred)
    rmse = np.sqrt(mean_squared_error(y[test_idx], pred))
    print(name, 'MAE:', round(mae, 2), 'RMSE:', round(rmse, 2))

Mean baseline MAE: 1.25 RMSE: 1.25
ANN MAE: 1.77 RMSE: 1.77
CNN MAE: 7.83 RMSE: 7.83
